In [2]:
# ======================================================
# Swin Transformer + Lightweight UNet++
# TRAIN metrics per epoch
# FINAL TEST metrics printed once
# DP-CORRECTED (80/20 split)
# ======================================================

import os, cv2, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score
)
from tqdm import tqdm
import albumentations as A

# -------------------------
# SEED & DEVICE
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------
# CONFIG
# -------------------------
IMG_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 50
NUM_WORKERS = 2

IMAGES_DIR = "kfu data/train images"
MASKS_DIR  = "kfu data/train masks"

# -------------------------
# AUGMENTATION (TRAIN ONLY)
# -------------------------
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9,1.1), rotate=(-20,20), translate_percent=0.05, p=0.5),
    A.RandomBrightnessContrast(p=0.4)
])

# ======================================================
# DATASET
# ======================================================
class FootUlcerDataset(Dataset):
    def __init__(self, ids, augment=False):
        self.ids = ids
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]

        img = cv2.imread(os.path.join(IMAGES_DIR, name + ".jpg"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        mask = cv2.imread(os.path.join(MASKS_DIR, name + ".png"), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)

        if self.augment:
            aug = train_aug(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        img = img / 255.0
        img = np.transpose(img, (2, 0, 1))

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        )

# -------------------------
# 80 / 20 SPLIT
# -------------------------
ids = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(".jpg")
    and os.path.exists(os.path.join(MASKS_DIR, os.path.splitext(f)[0] + ".png"))
])

train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=SEED)

train_ds = FootUlcerDataset(train_ids, augment=True)
test_ds  = FootUlcerDataset(test_ids, augment=False)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_ds)}")
print(f"Test samples : {len(test_ds)}")

# ======================================================
# MODEL: Swin Transformer + Lightweight UNet++
# ======================================================
class PatchEmbedding(nn.Module):
    def __init__(self, in_ch=3, embed_dim=64, patch=4):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, patch, patch)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x), (H, W)

class SwinBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        a,_ = self.attn(x,x,x)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x

class SwinUNetPP(nn.Module):
    def __init__(self, embed_dim=64, heads=4):
        super().__init__()
        self.embed = PatchEmbedding(3, embed_dim)
        self.encoder = nn.Sequential(
            SwinBlock(embed_dim, heads),
            SwinBlock(embed_dim, heads)
        )
        self.up = nn.Upsample(scale_factor=4, mode="bilinear", align_corners=True)
        self.out = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim//2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim//2, 1, 1)
        )

    def forward(self, x):
        x, (H, W) = self.embed(x)
        x = self.encoder(x)
        x = x.transpose(1,2).view(-1, x.shape[-1], H, W)
        return torch.sigmoid(self.out(self.up(x)))

# ======================================================
# LOSS (NaN-SAFE AMF)
# ======================================================
def amf_loss(pred, target, alpha=0.75):
    pred = torch.clamp(pred, 1e-7, 1-1e-7)
    pt = torch.where(target == 1, pred, 1 - pred)
    return (-torch.pow(1 - pt, alpha) * torch.log(pt)).mean()

# ======================================================
# METRICS
# ======================================================
def compute_all_metrics(P, M):
    P_bin = (P > 0.5).astype(np.uint8)

    acc  = accuracy_score(M, P_bin)
    prec = precision_score(M, P_bin, zero_division=0)
    rec  = recall_score(M, P_bin, zero_division=0)
    f1   = f1_score(M, P_bin, zero_division=0)

    inter = (P_bin * M).sum()
    dice = (2 * inter) / (P_bin.sum() + M.sum() + 1e-7)
    iou  = inter / (P_bin.sum() + M.sum() - inter + 1e-7)

    ap50 = average_precision_score(M, P_bin)
    map5095 = np.mean([
        average_precision_score(M, (P > t).astype(int))
        for t in np.arange(0.5, 1.0, 0.05)
    ])

    return acc, prec, rec, f1, dice, iou, ap50, map5095

# ======================================================
# TRAINING (TRAIN METRICS PER EPOCH)
# ======================================================
model = SwinUNetPP(embed_dim=64, heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("\nStarting training...\n")

for epoch in range(EPOCHS):
    model.train()
    train_preds, train_masks = [], []

    for imgs, masks in tqdm(train_loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = amf_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_preds.append(preds.detach().cpu())
        train_masks.append(masks.detach().cpu())

    P_tr = torch.cat(train_preds).numpy().flatten()
    M_tr = torch.cat(train_masks).numpy().flatten()
    tr = compute_all_metrics(P_tr, M_tr)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"TRAIN → Acc:{tr[0]:.4f} Prec:{tr[1]:.4f} Rec:{tr[2]:.4f} F1:{tr[3]:.4f} | "
        f"Dice:{tr[4]:.4f} IoU:{tr[5]:.4f} mAP50:{tr[6]:.4f} mAP50-95:{tr[7]:.4f}"
    )

# ======================================================
# FINAL TEST RESULTS (PRINT ONCE)
# ======================================================
model.eval()
test_preds, test_masks = [], []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        test_preds.append(preds.cpu())
        test_masks.append(masks.cpu())

P_te = torch.cat(test_preds).numpy().flatten()
M_te = torch.cat(test_masks).numpy().flatten()
te = compute_all_metrics(P_te, M_te)

print("\n================ FINAL TEST RESULTS ================")
print(f"Accuracy     : {te[0]:.4f}")
print(f"Precision    : {te[1]:.4f}")
print(f"Recall       : {te[2]:.4f}")
print(f"F1-score     : {te[3]:.4f}")
print(f"Dice         : {te[4]:.4f}")
print(f"IoU          : {te[5]:.4f}")
print(f"mAP@50       : {te[6]:.4f}")
print(f"mAP@50–95    : {te[7]:.4f}")
print("===================================================")

torch.save(model.state_dict(), "swin_unetpp_final.pth")
print("Model saved as swin_unetpp_final.pth")


Using device: cuda
Train samples: 1600
Test samples : 400

Starting training...



Epoch 01/50 | TRAIN → Acc:0.9758 Prec:0.2896 Rec:0.0159 F1:0.0302 | Dice:0.0302 IoU:0.0153 mAP50:0.0278 mAP50-95:0.0242


Epoch 02/50 | TRAIN → Acc:0.9767 Prec:0.5294 Rec:0.0607 F1:0.1089 | Dice:0.1089 IoU:0.0576 mAP50:0.0542 mAP50-95:0.0287


Epoch 03/50 | TRAIN → Acc:0.9767 Prec:0.5599 Rec:0.0803 F1:0.1405 | Dice:0.1405 IoU:0.0756 mAP50:0.0667 mAP50-95:0.0340


Epoch 04/50 | TRAIN → Acc:0.9768 Prec:0.5477 Rec:0.0866 F1:0.1496 | Dice:0.1496 IoU:0.0809 mAP50:0.0690 mAP50-95:0.0310


...

Epoch 46/50 | TRAIN → Acc:0.9814 Prec:0.6874 Rec:0.3849 F1:0.4935 | Dice:0.4935 IoU:0.3276 mAP50:0.2791 mAP50-95:0.1289


Epoch 47/50 | TRAIN → Acc:0.9818 Prec:0.6989 Rec:0.4058 F1:0.5135 | Dice:0.5135 IoU:0.3454 mAP50:0.2977 mAP50-95:0.1331


Epoch 48/50 | TRAIN → Acc:0.9820 Prec:0.7035 Rec:0.4121 F1:0.5197 | Dice:0.5197 IoU:0.3511 mAP50:0.3038 mAP50-95:0.1463


Epoch 49/50 | TRAIN → Acc:0.9828 Prec:0.7191 Rec:0.4397 F1:0.5457 | Dice:0.5457 IoU:0.3753 mAP50:0.3294 mAP50-95:0.1629


Epoch 50/50 | TRAIN → Acc:0.9828 Prec:0.7221 Rec:0.4404 F1:0.5471 | Dice:0.5471 IoU:0.3766 mAP50:0.3312 mAP50-95:0.1640

================ FINAL TEST RESULTS ================
Accuracy     : 0.9825
Precision    : 0.7405
Recall       : 0.4131
F1-score     : 0.5303
Dice         : 0.5303
IoU          : 0.3608
mAP@50       : 0.3199
mAP@50–95    : 0.1612
Model saved as swin_unetpp_final.pth


In [3]:
# ======================================================
# Swin Transformer + Lightweight UNet++
# TRAIN metrics per epoch
# FINAL TEST metrics printed once
# DP-CORRECTED (80/20 split)
# ======================================================

import os, cv2, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score
)
from tqdm import tqdm
import albumentations as A

# -------------------------
# SEED & DEVICE
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------
# CONFIG
# -------------------------
IMG_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 50
NUM_WORKERS = 2

IMAGES_DIR = "kfu data/train images"
MASKS_DIR  = "kfu data/train masks"

# -------------------------
# AUGMENTATION (TRAIN ONLY)
# -------------------------
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9,1.1), rotate=(-20,20), translate_percent=0.05, p=0.5),
    A.RandomBrightnessContrast(p=0.4)
])

# ======================================================
# DATASET
# ======================================================
class FootUlcerDataset(Dataset):
    def __init__(self, ids, augment=False):
        self.ids = ids
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]

        img = cv2.imread(os.path.join(IMAGES_DIR, name + ".jpg"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        mask = cv2.imread(os.path.join(MASKS_DIR, name + ".png"), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)

        if self.augment:
            aug = train_aug(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        img = img / 255.0
        img = np.transpose(img, (2, 0, 1))

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        )

# -------------------------
# 80 / 20 SPLIT
# -------------------------
ids = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(".jpg")
    and os.path.exists(os.path.join(MASKS_DIR, os.path.splitext(f)[0] + ".png"))
])

train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=SEED)

train_ds = FootUlcerDataset(train_ids, augment=True)
test_ds  = FootUlcerDataset(test_ids, augment=False)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_ds)}")
print(f"Test samples : {len(test_ds)}")

# ======================================================
# MODEL: Swin Transformer + Lightweight UNet++
# ======================================================
class PatchEmbedding(nn.Module):
    def __init__(self, in_ch=3, embed_dim=64, patch=4):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, patch, patch)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x), (H, W)

class SwinBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        a,_ = self.attn(x,x,x)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x

class SwinUNetPP(nn.Module):
    def __init__(self, embed_dim=64, heads=4):
        super().__init__()
        self.embed = PatchEmbedding(3, embed_dim)
        self.encoder = nn.Sequential(
            SwinBlock(embed_dim, heads),
            SwinBlock(embed_dim, heads)
        )
        self.up = nn.Upsample(scale_factor=4, mode="bilinear", align_corners=True)
        self.out = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim//2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim//2, 1, 1)
        )

    def forward(self, x):
        x, (H, W) = self.embed(x)
        x = self.encoder(x)
        x = x.transpose(1,2).view(-1, x.shape[-1], H, W)
        return torch.sigmoid(self.out(self.up(x)))

# ======================================================
# LOSS (NaN-SAFE AMF)
# ======================================================
def amf_loss(pred, target, alpha=0.75):
    pred = torch.clamp(pred, 1e-7, 1-1e-7)
    pt = torch.where(target == 1, pred, 1 - pred)
    return (-torch.pow(1 - pt, alpha) * torch.log(pt)).mean()

# ======================================================
# METRICS
# ======================================================
def compute_all_metrics(P, M):
    P_bin = (P > 0.5).astype(np.uint8)

    acc  = accuracy_score(M, P_bin)
    prec = precision_score(M, P_bin, zero_division=0)
    rec  = recall_score(M, P_bin, zero_division=0)
    f1   = f1_score(M, P_bin, zero_division=0)

    inter = (P_bin * M).sum()
    dice = (2 * inter) / (P_bin.sum() + M.sum() + 1e-7)
    iou  = inter / (P_bin.sum() + M.sum() - inter + 1e-7)

    ap50 = average_precision_score(M, P_bin)
    map5095 = np.mean([
        average_precision_score(M, (P > t).astype(int))
        for t in np.arange(0.5, 1.0, 0.05)
    ])

    return acc, prec, rec, f1, dice, iou, ap50, map5095

# ======================================================
# TRAINING (TRAIN METRICS PER EPOCH)
# ======================================================
model = SwinUNetPP(embed_dim=64, heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("\nStarting training...\n")

for epoch in range(EPOCHS):
    model.train()
    train_preds, train_masks = [], []

    for imgs, masks in tqdm(train_loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = amf_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_preds.append(preds.detach().cpu())
        train_masks.append(masks.detach().cpu())

    P_tr = torch.cat(train_preds).numpy().flatten()
    M_tr = torch.cat(train_masks).numpy().flatten()
    tr = compute_all_metrics(P_tr, M_tr)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"TRAIN → Acc:{tr[0]:.4f} Prec:{tr[1]:.4f} Rec:{tr[2]:.4f} F1:{tr[3]:.4f} | "
        f"Dice:{tr[4]:.4f} IoU:{tr[5]:.4f} mAP50:{tr[6]:.4f} mAP50-95:{tr[7]:.4f}"
    )

# ======================================================
# FINAL TEST RESULTS (PRINT ONCE)
# ======================================================
model.eval()
test_preds, test_masks = [], []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        test_preds.append(preds.cpu())
        test_masks.append(masks.cpu())

P_te = torch.cat(test_preds).numpy().flatten()
M_te = torch.cat(test_masks).numpy().flatten()
te = compute_all_metrics(P_te, M_te)

print("\n================ FINAL TEST RESULTS ================")
print(f"Accuracy     : {te[0]:.4f}")
print(f"Precision    : {te[1]:.4f}")
print(f"Recall       : {te[2]:.4f}")
print(f"F1-score     : {te[3]:.4f}")
print(f"Dice         : {te[4]:.4f}")
print(f"IoU          : {te[5]:.4f}")
print(f"mAP@50       : {te[6]:.4f}")
print(f"mAP@50–95    : {te[7]:.4f}")
print("===================================================")

torch.save(model.state_dict(), "swin_unetpp_final.pth")
print("Model saved as swin_unetpp_final.pth")


Using device: cuda
Train samples: 1600
Test samples : 400

Starting training...



Epoch 01/50 | TRAIN → Acc:0.9761 Prec:0.3609 Rec:0.0178 F1:0.0338 | Dice:0.0338 IoU:0.0172 mAP50:0.0296 mAP50-95:0.0244


Epoch 02/50 | TRAIN → Acc:0.9765 Prec:0.5338 Rec:0.0643 F1:0.1148 | Dice:0.1148 IoU:0.0609 mAP50:0.0565 mAP50-95:0.0287


Epoch 03/50 | TRAIN → Acc:0.9768 Prec:0.5736 Rec:0.0990 F1:0.1689 | Dice:0.1689 IoU:0.0922 mAP50:0.0783 mAP50-95:0.0364


Epoch 04/50 | TRAIN → Acc:0.9769 Prec:0.5820 Rec:0.0812 F1:0.1426 | Dice:0.1426 IoU:0.0768 mAP50:0.0690 mAP50-95:0.0306


...

Epoch 46/50 | TRAIN → Acc:0.9816 Prec:0.6932 Rec:0.3940 F1:0.5024 | Dice:0.5024 IoU:0.3355 mAP50:0.2875 mAP50-95:0.1302


Epoch 47/50 | TRAIN → Acc:0.9822 Prec:0.7076 Rec:0.4218 F1:0.5285 | Dice:0.5285 IoU:0.3592 mAP50:0.3121 mAP50-95:0.1442


Epoch 48/50 | TRAIN → Acc:0.9815 Prec:0.6942 Rec:0.3913 F1:0.5005 | Dice:0.5005 IoU:0.3338 mAP50:0.2861 mAP50-95:0.1367


Epoch 49/50 | TRAIN → Acc:0.9821 Prec:0.7093 Rec:0.4136 F1:0.5225 | Dice:0.5225 IoU:0.3536 mAP50:0.3072 mAP50-95:0.1450


Epoch 50/50 | TRAIN → Acc:0.9824 Prec:0.7173 Rec:0.4263 F1:0.5348 | Dice:0.5348 IoU:0.3650 mAP50:0.3194 mAP50-95:0.1581

================ FINAL TEST RESULTS ================
Accuracy     : 0.9819
Precision    : 0.7748
Recall       : 0.3424
F1-score     : 0.4749
Dice         : 0.4749
IoU          : 0.3114
mAP@50       : 0.2810
mAP@50–95    : 0.1240
Model saved as swin_unetpp_final.pth


In [4]:
# ======================================================
# Swin Transformer + Lightweight UNet++
# TRAIN metrics per epoch
# FINAL TEST metrics printed once
# DP-CORRECTED (80/20 split)
# ======================================================

import os, cv2, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score
)
from tqdm import tqdm
import albumentations as A

# -------------------------
# SEED & DEVICE
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------
# CONFIG
# -------------------------
IMG_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 50
NUM_WORKERS = 2

IMAGES_DIR = "kfu data/train images"
MASKS_DIR  = "kfu data/train masks"

# -------------------------
# AUGMENTATION (TRAIN ONLY)
# -------------------------
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9,1.1), rotate=(-20,20), translate_percent=0.05, p=0.5),
    A.RandomBrightnessContrast(p=0.4)
])

# ======================================================
# DATASET
# ======================================================
class FootUlcerDataset(Dataset):
    def __init__(self, ids, augment=False):
        self.ids = ids
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]

        img = cv2.imread(os.path.join(IMAGES_DIR, name + ".jpg"))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        mask = cv2.imread(os.path.join(MASKS_DIR, name + ".png"), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)

        if self.augment:
            aug = train_aug(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        img = img / 255.0
        img = np.transpose(img, (2, 0, 1))

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        )

# -------------------------
# 80 / 20 SPLIT
# -------------------------
ids = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(".jpg")
    and os.path.exists(os.path.join(MASKS_DIR, os.path.splitext(f)[0] + ".png"))
])

train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=SEED)

train_ds = FootUlcerDataset(train_ids, augment=True)
test_ds  = FootUlcerDataset(test_ids, augment=False)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_ds)}")
print(f"Test samples : {len(test_ds)}")

# ======================================================
# MODEL: Swin Transformer + Lightweight UNet++
# ======================================================
class PatchEmbedding(nn.Module):
    def __init__(self, in_ch=3, embed_dim=64, patch=4):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, patch, patch)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x), (H, W)

class SwinBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim*4),
            nn.GELU(),
            nn.Linear(dim*4, dim)
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        a,_ = self.attn(x,x,x)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x

class SwinUNetPP(nn.Module):
    def __init__(self, embed_dim=64, heads=4):
        super().__init__()
        self.embed = PatchEmbedding(3, embed_dim)
        self.encoder = nn.Sequential(
            SwinBlock(embed_dim, heads),
            SwinBlock(embed_dim, heads)
        )
        self.up = nn.Upsample(scale_factor=4, mode="bilinear", align_corners=True)
        self.out = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim//2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim//2, 1, 1)
        )

    def forward(self, x):
        x, (H, W) = self.embed(x)
        x = self.encoder(x)
        x = x.transpose(1,2).view(-1, x.shape[-1], H, W)
        return torch.sigmoid(self.out(self.up(x)))

# ======================================================
# LOSS (NaN-SAFE AMF)
# ======================================================
def amf_loss(pred, target, alpha=0.75):
    pred = torch.clamp(pred, 1e-7, 1-1e-7)
    pt = torch.where(target == 1, pred, 1 - pred)
    return (-torch.pow(1 - pt, alpha) * torch.log(pt)).mean()

# ======================================================
# METRICS
# ======================================================
def compute_all_metrics(P, M):
    P_bin = (P > 0.5).astype(np.uint8)

    acc  = accuracy_score(M, P_bin)
    prec = precision_score(M, P_bin, zero_division=0)
    rec  = recall_score(M, P_bin, zero_division=0)
    f1   = f1_score(M, P_bin, zero_division=0)

    inter = (P_bin * M).sum()
    dice = (2 * inter) / (P_bin.sum() + M.sum() + 1e-7)
    iou  = inter / (P_bin.sum() + M.sum() - inter + 1e-7)

    ap50 = average_precision_score(M, P_bin)
    map5095 = np.mean([
        average_precision_score(M, (P > t).astype(int))
        for t in np.arange(0.5, 1.0, 0.05)
    ])

    return acc, prec, rec, f1, dice, iou, ap50, map5095

# ======================================================
# TRAINING (TRAIN METRICS PER EPOCH)
# ======================================================
model = SwinUNetPP(embed_dim=64, heads=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("\nStarting training...\n")

for epoch in range(EPOCHS):
    model.train()
    train_preds, train_masks = [], []

    for imgs, masks in tqdm(train_loader, leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = amf_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_preds.append(preds.detach().cpu())
        train_masks.append(masks.detach().cpu())

    P_tr = torch.cat(train_preds).numpy().flatten()
    M_tr = torch.cat(train_masks).numpy().flatten()
    tr = compute_all_metrics(P_tr, M_tr)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"TRAIN → Acc:{tr[0]:.4f} Prec:{tr[1]:.4f} Rec:{tr[2]:.4f} F1:{tr[3]:.4f} | "
        f"Dice:{tr[4]:.4f} IoU:{tr[5]:.4f} mAP50:{tr[6]:.4f} mAP50-95:{tr[7]:.4f}"
    )

# ======================================================
# FINAL TEST RESULTS (PRINT ONCE)
# ======================================================
model.eval()
test_preds, test_masks = [], []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        test_preds.append(preds.cpu())
        test_masks.append(masks.cpu())

P_te = torch.cat(test_preds).numpy().flatten()
M_te = torch.cat(test_masks).numpy().flatten()
te = compute_all_metrics(P_te, M_te)

print("\n================ FINAL TEST RESULTS ================")
print(f"Accuracy     : {te[0]:.4f}")
print(f"Precision    : {te[1]:.4f}")
print(f"Recall       : {te[2]:.4f}")
print(f"F1-score     : {te[3]:.4f}")
print(f"Dice         : {te[4]:.4f}")
print(f"IoU          : {te[5]:.4f}")
print(f"mAP@50       : {te[6]:.4f}")
print(f"mAP@50–95    : {te[7]:.4f}")
print("===================================================")

torch.save(model.state_dict(), "swin_unetpp_final.pth")
print("Model saved as swin_unetpp_final.pth")


Using device: cuda
Train samples: 1600
Test samples : 400

Starting training...



Epoch 01/50 | TRAIN → Acc:0.9759 Prec:0.1672 Rec:0.0076 F1:0.0146 | Dice:0.0146 IoU:0.0073 mAP50:0.0245 mAP50-95:0.0235


Epoch 02/50 | TRAIN → Acc:0.9766 Prec:0.5400 Rec:0.0430 F1:0.0797 | Dice:0.0797 IoU:0.0415 mAP50:0.0457 mAP50-95:0.0277


Epoch 03/50 | TRAIN → Acc:0.9769 Prec:0.5919 Rec:0.0664 F1:0.1194 | Dice:0.1194 IoU:0.0635 mAP50:0.0613 mAP50-95:0.0317


Epoch 04/50 | TRAIN → Acc:0.9771 Prec:0.5996 Rec:0.0781 F1:0.1382 | Dice:0.1382 IoU:0.0742 mAP50:0.0685 mAP50-95:0.0331


...

Epoch 46/50 | TRAIN → Acc:0.9820 Prec:0.7048 Rec:0.4025 F1:0.5124 | Dice:0.5124 IoU:0.3444 mAP50:0.2977 mAP50-95:0.1424


Epoch 47/50 | TRAIN → Acc:0.9825 Prec:0.7046 Rec:0.4314 F1:0.5351 | Dice:0.5351 IoU:0.3653 mAP50:0.3173 mAP50-95:0.1435


Epoch 48/50 | TRAIN → Acc:0.9818 Prec:0.6980 Rec:0.3949 F1:0.5044 | Dice:0.5044 IoU:0.3372 mAP50:0.2898 mAP50-95:0.1393


Epoch 49/50 | TRAIN → Acc:0.9829 Prec:0.7181 Rec:0.4491 F1:0.5526 | Dice:0.5526 IoU:0.3818 mAP50:0.3355 mAP50-95:0.1642


Epoch 50/50 | TRAIN → Acc:0.9830 Prec:0.7210 Rec:0.4486 F1:0.5531 | Dice:0.5531 IoU:0.3823 mAP50:0.3364 mAP50-95:0.1682

================ FINAL TEST RESULTS ================
Accuracy     : 0.9822
Precision    : 0.7963
Recall       : 0.3397
F1-score     : 0.4762
Dice         : 0.4762
IoU          : 0.3125
mAP@50       : 0.2863
mAP@50–95    : 0.1291
Model saved as swin_unetpp_final.pth
